# Cross-Market Comparison: Houston · NYC · Miami · Chicago

**Goal:** Compare neighborhood-level forecasts across four major US markets using Census tabulation blocks and ZCTAs.

This notebook uses two resolution tiers:
- **Tabblock** (15-digit GEOID) for Houston and NYC, where jurisdiction-specific models exist
- **ZCTA** (5-digit ZIP) for nationwide coverage including Miami and Chicago via the ACS model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/homecastr/homecastr-cookbook/blob/main/examples/neighborhoods/02_market_comparison.ipynb)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from homecastr import HomecastrClient

client = HomecastrClient(os.getenv("HOMECASTR_API_KEY", "hc_demo_public_readonly"))

## Part 1: High-Resolution — Houston & NYC Tabblocks

Tabblocks are the finest available geography (~city block level). We have 34K Houston tabblocks and 32K NYC tabblocks with full P10–P90 fan charts.

In [ ]:
# Real tabblock GEOIDs from Homecastr database
TABBLOCKS = {
    # NYC — Manhattan
    "NYC: Staten Island (360050002000001)": "360050002000001",
    "NYC: Staten Island (360050002001000)": "360050002001000",
    "NYC: Staten Island (360050002001001)": "360050002001001",
    # Houston
    "Houston (480717106001022)": "480717106001022",
    "Houston (481576704001000)": "481576704001000",
    "Houston (481576704001001)": "481576704001001",
}

df_tab = client.forecast.by_tabblock.retrieve_many(list(TABBLOCKS.values()))
df_tab.insert(0, "label", list(TABBLOCKS.keys()))
df_tab["city"] = df_tab["label"].apply(lambda x: "NYC" if x.startswith("NYC") else "Houston")
print(df_tab[["label", "current_value", "p50", "growth_pct", "jurisdiction"]].to_string(index=False))

## Part 2: Nationwide ZCTA Coverage

ZCTAs cover ~20K ZIP codes nationwide via the ACS model. Use these for markets without jurisdiction-specific models yet.

In [ ]:
ZCTAS = {
    "Houston Galleria (77056)": "77056",
    "Houston Heights (77008)": "77008",
    "Manhattan Midtown (10001)": "10001",
    "Manhattan Upper West (10024)": "10024",
    "Miami Beach (33139)": "33139",
    "Miami Brickell (33131)": "33131",
    "Chicago Lincoln Park (60614)": "60614",
    "Chicago Wicker Park (60622)": "60622",
    "SF Mission (94110)": "94110",
    "SF Pacific Heights (94115)": "94115",
    "Seattle Capitol Hill (98102)": "98102",
    "Seattle Ballard (98107)": "98107",
}

df_zcta = client.forecast.by_zcta.retrieve_many(list(ZCTAS.values()))
df_zcta.insert(0, "label", list(ZCTAS.keys()))
df_zcta["city"] = df_zcta["label"].str.split(" ").str[0]

print(df_zcta[["label", "current_value", "p50_12m", "p50_60m", "jurisdiction"]].to_string(index=False))

In [ ]:
CITY_COLORS = {
    "Houston": "steelblue",
    "Manhattan": "crimson",
    "Miami": "darkorange",
    "Chicago": "forestgreen",
    "SF": "mediumpurple",
    "Seattle": "teal",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Cross-Market Comparison via ZCTA Forecasts", fontsize=14, fontweight="bold")

colors = [CITY_COLORS.get(r["city"], "gray") for _, r in df_zcta.iterrows()]

# 12-month forecast
df_s = df_zcta.sort_values("p50_12m", ascending=True)
axes[0].barh(df_s["label"], df_s["p50_12m"] / 1e3,
             color=[CITY_COLORS.get(r["city"], "gray") for _, r in df_s.iterrows()])
axes[0].set_xlabel("12-Month Median Forecast ($000s)")
axes[0].set_title("1-Year Outlook")

# 60-month forecast
df_s2 = df_zcta.sort_values("p50_60m", ascending=True)
axes[1].barh(df_s2["label"], df_s2["p50_60m"] / 1e3,
             color=[CITY_COLORS.get(r["city"], "gray") for _, r in df_s2.iterrows()])
axes[1].set_xlabel("60-Month Median Forecast ($000s)")
axes[1].set_title("5-Year Outlook")

legend = [Patch(facecolor=c, label=city) for city, c in CITY_COLORS.items()]
fig.legend(handles=legend, loc="lower center", ncol=6, frameon=False)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

## Export

In [ ]:
df_zcta.to_csv("cross_market_zcta_forecasts.csv", index=False)
df_tab.to_csv("nyc_houston_tabblock_forecasts.csv", index=False)
print(f"Saved {len(df_zcta)} ZCTA rows and {len(df_tab)} tabblock rows.")